In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path


# ============================================================
# 1. ARQUIVOS
# ============================================================

arquivo_microdados = Path(
    "01_PENSE2024_MICRODADOS.parquet"
)

arquivo_recomendadas = Path(
    "01_Colunas_recomendadas_pense_2024_final.xlsx"
)

arquivo_saida = Path(
    "02_PENSE2024_MICRODADOS_FILTRADO.parquet"
)


# ============================================================
# 2. LER A PLANILHA DE COLUNAS RECOMENDADAS
# ============================================================

df_recomendadas = pd.read_excel(
    arquivo_recomendadas,
    header=1
)


# ============================================================
# 3. VERIFICAR COLUNAS DA PLANILHA
# ============================================================

print("Colunas encontradas no Excel:")

print(
    df_recomendadas.columns.tolist()
)


# ============================================================
# 4. PEGAR OS CÓDIGOS RECOMENDADOS
# ============================================================

codigos_recomendados = set(
    df_recomendadas["Código da coluna"]
    .dropna()
    .astype(str)
    .str.strip()
)


# ============================================================
# 5. LER O PARQUET
# ============================================================

df = pd.read_parquet(
    arquivo_microdados
)


# ============================================================
# 6. IDENTIFICAR AS COLUNAS QUE DEVEM SER MANTIDAS
# ============================================================

colunas_mantidas = [
    coluna
    for coluna in df.columns
    if str(coluna).strip() in codigos_recomendados
]


# ============================================================
# 7. MOSTRAR RESULTADO
# ============================================================

print()
print("=" * 80)

print(
    f"Total de colunas no arquivo: {len(df.columns)}"
)

print(
    f"Colunas recomendadas: {len(codigos_recomendados)}"
)

print(
    f"Colunas mantidas: {len(colunas_mantidas)}"
)

print("=" * 80)


# ============================================================
# 8. SELECIONAR SOMENTE AS COLUNAS RECOMENDADAS
# ============================================================

df = df[colunas_mantidas].copy()


# ============================================================
# 9. CONVERTER TODAS AS COLUNAS PARA STRING
# ============================================================

for coluna in df.columns:

    # Primeiro transforma valores como:
    #
    # 1.0  → 1
    # 2.0  → 2
    # -9.0 → -9
    #
    # somente quando forem números inteiros.

    def normalizar(valor):

        if pd.isna(valor):
            return pd.NA

        try:

            numero = float(valor)

            if numero.is_integer():
                return str(int(numero))

        except (ValueError, TypeError):
            pass

        return str(valor).strip()


    df[coluna] = df[coluna].map(normalizar).astype("string")


# ============================================================
# 10. MOSTRAR OS TIPOS
# ============================================================

print()
print("Tipos das colunas após conversão:")
print()

print(df.dtypes)


# ============================================================
# 11. SALVAR
# ============================================================

# Garantir que todas as colunas sejam armazenadas como string
for coluna in df.columns:
    df[coluna] = df[coluna].astype("string")

df.to_parquet(
    arquivo_saida,
    engine="pyarrow",
    compression="snappy",
    index=False
)


# ============================================================
# 12. VERIFICAÇÃO
# ============================================================

print()
print("=" * 80)
print("ARQUIVO CRIADO")
print("=" * 80)

print(
    arquivo_saida
)

print(
    f"Linhas: {len(df):,}"
)

print(
    f"Colunas: {len(df.columns)}"
)

Colunas encontradas no Excel:
['Nº', 'Dimensão', 'Código da coluna', 'Descrição da variável', 'Prioridade', 'Uso recomendado', 'Nível', 'Justificativa', 'Cuidados metodológicos', 'Tipo', 'Categorias/codificação', 'Observações do dicionário', 'Nova numeracao', 'Nova codificacao', 'Percentagem nulos']

Total de colunas no arquivo: 430
Colunas recomendadas: 41
Colunas mantidas: 41

Tipos das colunas após conversão:

REGIAO         string
UF             string
CAPITAL        string
B01001A        string
IDADE_AGREG    string
B01002         string
B01006         string
B01007         string
B01015B        string
B01016         string
B03006B        string
B03010C        string
B05002A        string
B07002         string
B07004         string
B07005         string
B07006         string
B07007B        string
B07011A        string
B07012A        string
B07013         string
B07009         string
B07016         string
B12003         string
B12004         string
B12005         string
B12006     

In [2]:
import pandas as pd

# Ler o arquivo Parquet
df = pd.read_parquet("02_PENSE2024_MICRODADOS_FILTRADO.parquet")

# Imprimir os códigos em ordem alfabética
for coluna in sorted(df.columns):
    print(coluna)

# Total de colunas
print("\nTotal:", len(df.columns))

B01001A
B01002
B01006
B01007
B01015B
B01016
B03006B
B03010C
B05002A
B07002
B07004
B07005
B07006
B07007B
B07009
B07011A
B07012A
B07013
B07016
B09001
B09002
B09003A
B09010A1
B09016A1
B09016A2
B12003
B12004
B12005
B12006
B12007
B12008
B12009
CAPITAL
E01P117
E01P11801
E01P11802
E01P8505
E01P8510
IDADE_AGREG
REGIAO
UF

Total: 41


In [3]:
import pandas as pd
from IPython.display import HTML, display


# ============================================================
# 1. CARREGAR O DICIONÁRIO
# ============================================================

dicionario = pd.read_excel(
    "00_Dicionário_Microdados_PENSE2024.xlsx",
    engine="openpyxl"
)


# ============================================================
# 2. CARREGAR AS REGRAS DE NOVA NUMERAÇÃO
# ============================================================

regras = pd.read_excel(
    "01_Colunas_recomendadas_pense_2024_final.xlsx",
    header=1
)


# ============================================================
# 3. NORMALIZAR TEXTO
# ============================================================

def normalizar(texto):

    return "".join(
        str(texto).split()
    ).lower()


# ============================================================
# 4. CRIAR DICIONÁRIO ORIGINAL
# ============================================================

dico_map = {}


for _, row in dicionario.iterrows():

    desc = str(
        row["Descrição da Variável"]
    ).strip()

    if not desc or desc == "nan":
        continue

    chave = normalizar(desc)

    cat = (
        str(row["Categoria"])
        if pd.notna(row["Categoria"])
        else ""
    )

    cat_dict = {}

    if cat:

        for linha in cat.split("\n"):

            linha = linha.strip()

            if " - " not in linha:
                continue

            codigo, significado = linha.split(
                " - ",
                1
            )

            codigo = codigo.strip()
            significado = significado.strip()

            cat_dict[codigo] = significado

            # 1 -> 1
            try:
                cat_dict[
                    str(int(float(codigo)))
                ] = significado
            except:
                pass

            # 1 -> 1.0
            try:
                cat_dict[
                    str(float(codigo))
                ] = significado
            except:
                pass

    dico_map[chave] = cat_dict


# ============================================================
# 5. FUNÇÃO PARA INTERPRETAR A NOVA NUMERAÇÃO
# ============================================================

def criar_mapa_numeracao(regra):

    mapa = {}

    if pd.isna(regra):
        return mapa

    regra = str(regra).strip()

    regra = regra.rstrip(")")


    for parte in regra.split(","):

        parte = parte.strip()

        if "=" not in parte:
            continue

        origem, destino = parte.split(
            "=",
            1
        )

        destino = destino.strip()

        for codigo in origem.split("/"):

            codigo = codigo.strip()

            if codigo:
                mapa[codigo] = destino

    return mapa


# ============================================================
# 6. CRIAR MAPA:
#
# CÓDIGO DA COLUNA → REGRA
# ============================================================

mapa_regras = {}


for _, row in regras.iterrows():

    codigo = row["Código da coluna"]

    regra = row["Nova numeracao"]

    if pd.isna(codigo) or pd.isna(regra):
        continue

    mapa_regras[
        str(codigo).strip()
    ] = str(regra).strip()


# ============================================================
# 7. GERAR TABELA FINAL
# ============================================================

tabela_dados = []


for col in df.columns:

    col_limpa = str(col).strip()


    # ========================================================
    # Separar código e descrição da variável
    # ========================================================

    if " - " in col_limpa:

        codigo_variavel, descricao_variavel = (
            col_limpa.split(
                " - ",
                1
            )
        )

        codigo_variavel = codigo_variavel.strip()
        descricao_variavel = descricao_variavel.strip()

    else:

        codigo_variavel = col_limpa
        descricao_variavel = col_limpa


    # ========================================================
    # Dicionário original da variável
    # ========================================================

    chave_busca = normalizar(
        descricao_variavel
    )

    cat_dict = dico_map.get(
        chave_busca,
        {}
    )


    # ========================================================
    # REGRA DE NOVA NUMERAÇÃO
    # ========================================================

    regra = mapa_regras.get(
        codigo_variavel
    )


    mapa_novo = criar_mapa_numeracao(
        regra
    )


    # ========================================================
    # CRIAR DESCRIÇÃO DOS NOVOS CÓDIGOS
    # ========================================================

    descricao_nova = {}


    for codigo_original, codigo_novo in mapa_novo.items():

        rotulo_original = cat_dict.get(
            codigo_original
        )


        if rotulo_original is None:

            try:

                rotulo_original = cat_dict.get(
                    str(float(codigo_original))
                )

            except:

                rotulo_original = None


        if rotulo_original is None:

            rotulo_original = codigo_original


        # Se vários códigos antigos viraram o mesmo
        # código novo, juntar as descrições.

        if codigo_novo not in descricao_nova:

            descricao_nova[codigo_novo] = []

        descricao_nova[codigo_novo].append(
            rotulo_original
        )


    # ========================================================
    # CONTAR AS RESPOSTAS APÓS A NOVA NUMERAÇÃO
    # ========================================================

    counts = df[col].value_counts(
        dropna=False
    )

    total = len(df)

    respostas_formatadas = []


    for cod, qtd in counts.items():

        pct = (
            qtd / total
        ) * 100


        # ----------------------------------------------------
        # NaN
        # ----------------------------------------------------

        if pd.isna(cod):

            respostas_formatadas.append(
                f"[NaN] Valores Nulos / Em Branco (NaN): "
                f"{pct:.2f}%"
            )

            continue


        # ----------------------------------------------------
        # Normalizar código
        # ----------------------------------------------------

        cod_str = str(cod).strip()

        if cod_str.endswith(".0"):
            cod_str = cod_str[:-2]


        # ----------------------------------------------------
        # Descrição do novo código
        # ----------------------------------------------------

        descricoes = descricao_nova.get(
            cod_str,
            []
        )


        if descricoes:

            descricao = " / ".join(
                dict.fromkeys(descricoes)
            )

            respostas_formatadas.append(
                f"[{cod_str}] {descricao}: "
                f"{pct:.2f}%"
            )

        else:

            respostas_formatadas.append(
                f"[{cod_str}]: "
                f"{pct:.2f}%"
            )


    # ========================================================
    # ADICIONAR À TABELA
    # ========================================================

    tabela_dados.append({

        "Pergunta / Variável":
            col_limpa,

        "Distribuição das Respostas":
            "\n".join(
                respostas_formatadas
            )
    })


# ============================================================
# 8. DATAFRAME FINAL
# ============================================================

df_tabela = pd.DataFrame(
    tabela_dados
)


# ============================================================
# 9. ESTILIZAÇÃO
# ============================================================

styler = df_tabela.style.set_properties(
    **{
        "white-space": "pre-wrap",
        "text-align": "left"
    }
)


# ============================================================
# 10. EXIBIR
# ============================================================

print(
    "========== DISTRIBUIÇÃO APÓS A NOVA NUMERAÇÃO ==========\n"
)

display(
    HTML(
        styler.to_html()
    )
)

========== DISTRIBUIÇÃO APÓS A NOVA NUMERAÇÃO ==========



,Pergunta / Variável,Distribuição das Respostas
0,REGIAO,[2] 2: 33.29% [1] 1: 24.69% [3] 3: 18.56% [5] 5: 12.11% [4] 4: 11.35%
1,UF,[32]: 4.87% [35]: 4.81% [33]: 4.62% [26]: 4.59% [29]: 4.52% [11]: 4.32% [31]: 4.26% [41]: 4.11% [42]: 4.04% [17]: 4.04% [13]: 3.95% [50]: 3.92% [21]: 3.85% [28]: 3.83% [27]: 3.82% [15]: 3.51% [25]: 3.44% [22]: 3.31% [12]: 3.23% [43]: 3.20% [16]: 3.10% [23]: 2.97% [24]: 2.95% [51]: 2.94% [52]: 2.84% [14]: 2.54% [53]: 2.42%
2,CAPITAL,[1] 1: 51.41% [0]: 48.59%
3,B01001A,[2] 2: 48.38% [1] 1: 48.28% [NaN] Valores Nulos / Em Branco (NaN): 3.34%
4,IDADE_AGREG,[2] 2: 49.89% [3] 3: 26.83% [1] 1: 15.50% [4] 4: 4.37% [NaN] Valores Nulos / Em Branco (NaN): 3.34% [-9]: 0.08%
5,B01002,[4] 4: 41.79% [1] 1: 39.18% [2] 2: 10.31% [NaN] Valores Nulos / Em Branco (NaN): 3.34% [3] 3: 2.66% [5] 5: 2.17% [-9]: 0.55% [-2]: 0.01%
6,B01006,[1] 2: 86.93% [2]: 9.69% [NaN] Valores Nulos / Em Branco (NaN): 3.34% [-9]: 0.03% [-2]: 0.01%
7,B01007,[1] 2: 61.33% [2]: 35.25% [NaN] Valores Nulos / Em Branco (NaN): 3.34% [-9]: 0.06% [-2]: 0.01%
8,B01015B,[1] 2: 61.15% [2]: 35.45% [NaN] Valores Nulos / Em Branco (NaN): 3.34% [-9]: 0.04% [-2]: 0.02%
9,B01016,[1] 2: 94.01% [NaN] Valores Nulos / Em Branco (NaN): 3.34% [2]: 2.60% [-9]: 0.03% [-2]: 0.02%
